# Prework - Zpracování kategorických dat (Processing Categorical Data)

Modely strojového učení dokáží pracovat výhradně s numerickými hodnotami. V tomto cvičení navazujeme na předchozí krok čištění dat a převedeme všechny kategorické proměnné na binární indikátory (tzv. dummy variables / One-Hot Encoding).

### Cíle cvičení:
1. **Načtení vyčištěného datasetu** (`heart_data_exercise_1.csv`).
2. **Identifikace všech kategorických proměnných** (textových i diskrétních kódů EKG vyšetření).
3. **Převod kategorií na čísla** pomocí metody `pd.get_dummies(..., drop_first=True)`.
4. **Uložení nového datasetu** do souboru `heart_data_exercise_2.csv` bez sloupce s indexem.

## 1. Načtení vyčištěného datasetu z předchozího kroku

In [1]:
import os
import pandas as pd

input_path = os.path.join("data", "heart_data_exercise_1.csv")
df = pd.read_csv(input_path)

print(f"Rozměry vyčištěného datasetu: {df.shape[0]} řádků, {df.shape[1]} sloupců")
df.head()

Rozměry vyčištěného datasetu: 303 řádků, 14 sloupců


,age,sex,chestpain,restbp,chol,fbs,restecg,maxhr,exang,oldpeak,slope,ca,thal,ahd
0,63,male,typical,145,233.0,yes,2,150,no,2.3,3,0.0,fixed,no
1,67,male,asymptomatic,160,286.0,no,2,108,yes,1.5,2,3.0,normal,yes
2,67,male,asymptomatic,120,229.0,no,2,129,yes,2.6,2,2.0,reversable,yes
3,37,male,nonanginal,130,240.0,no,0,187,no,3.5,3,0.0,reversable,no
4,41,female,nontypical,130,204.0,no,2,172,no,1.4,1,0.0,fixed,no


## 2. Identifikace kategorických proměnných

Identifikujeme všechny proměnné s kategoriálním charakterem:
- **Textové (nominální) proměnné:**
  - `sex`: pohlaví (`female`, `male`)
  - `chestpain`: typ bolesti na hrudi (`asymptomatic`, `nonanginal`, `nontypical`, `typical`)
  - `fbs`: hladina cukru nalačno (`no`, `yes`)
  - `exang`: cvičením indukovaná angina (`no`, `yes`)
  - `thal`: thalasémie (`fixed`, `normal`, `reversable`)
  - `ahd`: srdeční onemocnění – cílová proměnná (`no`, `yes`)
- **Diskrétní proměnné s kategoriálním charakterem:**
  - `restecg`: výsledek klidového EKG (kategorie `0`, `1`, `2`)

In [2]:
cat_cols = ["sex", "chestpain", "fbs", "restecg", "exang", "thal", "ahd"]

print("Přehled unikátních hodnot v kategorických sloupcích:")
for col in cat_cols:
    unique_values = sorted(df[col].unique())
    print(f"- {col:12s} ({len(unique_values)} hodnot): {unique_values}")

Přehled unikátních hodnot v kategorických sloupcích:
- sex          (2 hodnot): ['female', 'male']
- chestpain    (4 hodnot): ['asymptomatic', 'nonanginal', 'nontypical', 'typical']
- fbs          (2 hodnot): ['no', 'yes']
- restecg      (3 hodnot): [0, 1, 2]
- exang        (2 hodnot): ['no', 'yes']
- thal         (3 hodnot): ['fixed', 'normal', 'reversable']
- ahd          (2 hodnot): ['no', 'yes']


## 3. Převod textových hodnot na numerické (One-Hot Encoding)

Použijeme metodu `pd.get_dummies()` s parametrem `drop_first=True`:
- Pro $k$ kategorií se vytvoří $k - 1$ binárních sloupců.
- První kategorie slouží jako referenční báze (když mají všechny ostatní sloupce hodnotu `0`, znamená to výskyt první kategorie).
- Tím se předchází multikolinearitě (**Dummy Variable Trap**).

In [3]:
# Zakódování kategorických proměnných
encoded_df = pd.get_dummies(
    df,
    columns=cat_cols,
    drop_first=True,
    dtype=int
)

print(f"Původní počet sloupců: {df.shape[1]}")
print(f"Nový počet sloupců po kódování: {encoded_df.shape[1]}")
print("\nSeznam všech sloupců po transformaci:")
print(encoded_df.columns.tolist())

Původní počet sloupců: 14
Nový počet sloupců po kódování: 18

Seznam všech sloupců po transformaci:
['age', 'restbp', 'chol', 'maxhr', 'oldpeak', 'slope', 'ca', 'sex_male', 'chestpain_nonanginal', 'chestpain_nontypical', 'chestpain_typical', 'fbs_yes', 'restecg_1', 'restecg_2', 'exang_yes', 'thal_normal', 'thal_reversable', 'ahd_yes']


## 4. Inspekce zakódovaného DataFrame

Ověříme, že všechny sloupce mají nyní numerický datový typ (`int` nebo `float`).

In [4]:
print("Zastoupení datových typů:")
print(encoded_df.dtypes.value_counts())

encoded_df.head(10)

Zastoupení datových typů:
int32      11
int64       4
float64     3
Name: count, dtype: int64


,age,restbp,chol,maxhr,oldpeak,slope,ca,sex_male,chestpain_nonanginal,chestpain_nontypical,chestpain_typical,fbs_yes,restecg_1,restecg_2,exang_yes,thal_normal,thal_reversable,ahd_yes
0,63,145,233.0,150,2.3,3,0.0,1,0,0,1,1,0,1,0,0,0,0
1,67,160,286.0,108,1.5,2,3.0,1,0,0,0,0,0,1,1,1,0,1
2,67,120,229.0,129,2.6,2,2.0,1,0,0,0,0,0,1,1,0,1,1
3,37,130,240.0,187,3.5,3,0.0,1,1,0,0,0,0,0,0,0,1,0
4,41,130,204.0,172,1.4,1,0.0,0,0,1,0,0,0,1,0,0,0,0
5,56,120,236.0,178,0.8,1,0.0,1,0,1,0,0,0,0,0,1,0,0
6,62,140,268.0,160,3.6,3,2.0,0,0,0,0,0,0,1,0,1,0,1
7,57,120,354.0,163,0.6,1,0.0,0,0,0,0,0,0,0,1,1,0,0
8,63,130,254.0,147,1.4,2,1.0,1,0,0,0,0,0,1,0,0,1,1
9,53,140,203.0,155,3.1,3,0.0,1,0,0,0,1,0,1,1,0,1,1


## 5. Uložení výsledného datasetu do souboru .csv bez indexu

Dataset uložíme jako `heart_data_exercise_2.csv` s parametrem `index=False`.

In [5]:
output_path = os.path.join("data", "heart_data_exercise_2.csv")
encoded_df.to_csv(output_path, index=False)

print(f"Dataset úspěšně uložen do: '{output_path}'")
print(f"Finální rozměry: {encoded_df.shape[0]} řádků, {encoded_df.shape[1]} sloupců")

Dataset úspěšně uložen do: 'data\heart_data_exercise_2.csv'
Finální rozměry: 303 řádků, 18 sloupců
